# Phase 7 — Production Face Search

## Objective

The goal of this phase is to build a reusable face-search engine
that can search registered missing-person records stored in the
SQLite database.

The system will:

- Accept a query photograph
- Detect valid faces using MTCNN
- Generate a 512-dimensional ArcFace embedding
- Retrieve stored embeddings from SQLite
- Calculate cosine distances
- Rank candidates at the person level
- Apply a similarity threshold
- Return potential matches for human verification

The search functionality developed in this phase will later serve
as the core face-search component of the application backend.

## Production Search Architecture

```text
Query Photograph
       |
       v
MTCNN Face Detection
       |
       v
ArcFace Embedding
       |
       v
SQLite Database
       |
       v
Stored Face Embeddings
       |
       v
Cosine Distance
       |
       v
Person-Level Ranking
       |
       v
Threshold Decision
       |
       v
Potential Matches
       |
       v
Human Verification

In [2]:
import sqlite3
import numpy as np
from deepface import DeepFace

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
DATABASE_PATH = "missing_persons.db"

connection = sqlite3.connect(DATABASE_PATH)

cursor = connection.cursor()

print("Database connection successful!")

Database connection successful!


In [4]:
# Check registered persons

cursor.execute("""
SELECT person_id, name, status
FROM persons
""")

persons = cursor.fetchall()

print("Registered persons:", len(persons))

for person in persons:
    print(person)

Registered persons: 2
(1, 'Test Person', 'MISSING')
(2, 'Test Person 2', 'MISSING')


In [5]:
# Load all face embeddings from the database

cursor.execute("""
SELECT
    persons.person_id,
    persons.name,
    photos.photo_id,
    photos.file_path,
    embeddings.embedding,
    embeddings.model_name
FROM embeddings
JOIN photos
    ON embeddings.photo_id = photos.photo_id
JOIN persons
    ON photos.person_id = persons.person_id
""")

database_records = cursor.fetchall()

print("Embedding records:", len(database_records))

for record in database_records:

    embedding = np.frombuffer(
        record[4],
        dtype=np.float64
    )

    print(
        "Person:", record[1],
        "| Photo:", record[2],
        "| Embedding shape:", embedding.shape,
        "| Model:", record[5]
    )

Embedding records: 3
Person: Test Person | Photo: 1 | Embedding shape: (512,) | Model: ArcFace
Person: Test Person 2 | Photo: 2 | Embedding shape: (512,) | Model: ArcFace
Person: Test Person | Photo: 3 | Embedding shape: (512,) | Model: ArcFace


In [6]:
def load_face_database():

    cursor.execute("""
    SELECT
        persons.person_id,
        persons.name,
        photos.photo_id,
        photos.file_path,
        embeddings.embedding,
        embeddings.model_name
    FROM embeddings
    JOIN photos
        ON embeddings.photo_id = photos.photo_id
    JOIN persons
        ON photos.person_id = persons.person_id
    """)

    records = cursor.fetchall()

    database = []

    for record in records:

        embedding = np.frombuffer(
            record[4],
            dtype=np.float64
        )

        database.append({
            "person_id": record[0],
            "name": record[1],
            "photo_id": record[2],
            "file_path": record[3],
            "embedding": embedding,
            "model_name": record[5]
        })

    return database


face_database = load_face_database()

print("Face database loaded!")
print("Total embeddings:", len(face_database))

Face database loaded!
Total embeddings: 3


In [7]:
def cosine_distance(embedding_a, embedding_b):

    similarity = np.dot(embedding_a, embedding_b) / (
        np.linalg.norm(embedding_a) *
        np.linalg.norm(embedding_b)
    )

    return 1 - similarity


print("Cosine distance function ready!")

Cosine distance function ready!


In [11]:
def detect_faces(image_path):

    detections = DeepFace.extract_faces(
        img_path=image_path,
        detector_backend="mtcnn",
        enforce_detection=False,
        align=True
    )

    valid_faces = [
        detection
        for detection in detections
        if detection["confidence"] > 0
    ]

    return valid_faces


def generate_face_embeddings(image_path):

    faces = detect_faces(image_path)

    embeddings = []

    for face in faces:

        face_crop = face["face"]

        embedding = DeepFace.represent(
            img_path=face_crop,
            model_name="ArcFace",
            detector_backend="skip",
            enforce_detection=False
        )[0]["embedding"]

        embeddings.append(np.array(embedding))

    return embeddings


print("Face processing functions ready!")

Face processing functions ready!


In [12]:
def search_missing_person(image_path, top_k=5, threshold=0.68):

    # Generate query embedding
    query_embeddings = generate_face_embeddings(image_path)

    # No face detected
    if len(query_embeddings) == 0:
        return {
            "status": "NO_FACE",
            "results": []
        }

    # More than one face detected
    if len(query_embeddings) > 1:
        return {
            "status": "MULTIPLE_FACES",
            "results": []
        }

    # Get the single query embedding
    query_embedding = query_embeddings[0]

    # Load latest database
    face_database = load_face_database()

    if len(face_database) == 0:
        return {
            "status": "NO_DATABASE_RECORDS",
            "results": []
        }

    # Compare query against every stored embedding
    person_results = {}

    for record in face_database:

        distance = cosine_distance(
            query_embedding,
            record["embedding"]
        )

        person_id = record["person_id"]

        if person_id not in person_results:

            person_results[person_id] = {
                "person_id": person_id,
                "name": record["name"],
                "best_distance": distance,
                "best_photo_id": record["photo_id"],
                "reference_photos": 1
            }

        else:

            person_results[person_id]["reference_photos"] += 1

            if distance < person_results[person_id]["best_distance"]:

                person_results[person_id]["best_distance"] = distance
                person_results[person_id]["best_photo_id"] = record["photo_id"]

    # Rank people by their best matching reference photo
    results = list(person_results.values())

    results.sort(
        key=lambda x: x["best_distance"]
    )

    results = results[:top_k]

    # Make the decision
    if results[0]["best_distance"] < threshold:
        status = "POTENTIAL_MATCH"
    else:
        status = "NO_RELIABLE_MATCH"

    return {
        "status": status,
        "results": results
    }


print("Production search function ready!")

Production search function ready!


In [13]:
result = search_missing_person(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

print("Search Status:", result["status"])
print()

for rank, person in enumerate(result["results"], start=1):

    print(
        f"{rank}. {person['name']} "
        f"| Best Distance: {person['best_distance']:.4f} "
        f"| Reference Photos: {person['reference_photos']}"
    )

Search Status: POTENTIAL_MATCH

1. Test Person | Best Distance: 0.2426 | Reference Photos: 2
2. Test Person 2 | Best Distance: 0.9581 | Reference Photos: 1


In [14]:
# Test the production search with an image containing no face

result_no_face = search_missing_person(
    "test_no_face.jpg",
    top_k=5
)

print("Search Status:", result_no_face["status"])
print("Number of Results:", len(result_no_face["results"]))

Search Status: NO_FACE
Number of Results: 0


In [15]:
# Test the production search with an image containing multiple faces

result_multiple_faces = search_missing_person(
    "test_multiple_faces.jpg",
    top_k=5
)

print("Search Status:", result_multiple_faces["status"])
print("Number of Results:", len(result_multiple_faces["results"]))

Search Status: MULTIPLE_FACES
Number of Results: 0


In [16]:
# Test an unknown person

result_unknown = search_missing_person(
    "deepface_repo/tests/unit/dataset/img13.jpg",
    top_k=5
)

print("Search Status:", result_unknown["status"])
print()

for rank, person in enumerate(result_unknown["results"], start=1):

    print(
        f"{rank}. {person['name']} "
        f"| Best Distance: {person['best_distance']:.4f} "
        f"| Reference Photos: {person['reference_photos']}"
    )

Search Status: NO_RELIABLE_MATCH

1. Test Person 2 | Best Distance: 0.8789 | Reference Photos: 1
2. Test Person | Best Distance: 1.0446 | Reference Photos: 2


In [17]:
def display_search_result(result):

    print("Search Status:", result["status"])
    print()

    if len(result["results"]) == 0:
        print("No candidates available.")
        return

    print("Potential Candidates:\n")

    for rank, person in enumerate(result["results"], start=1):

        print(f"Rank {rank}")
        print(f"Name: {person['name']}")
        print(f"Best Distance: {person['best_distance']:.4f}")
        print(f"Best Photo ID: {person['best_photo_id']}")
        print(f"Reference Photos: {person['reference_photos']}")
        print("-" * 40)


print("Result display function ready!")

Result display function ready!


In [18]:
result = search_missing_person(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

display_search_result(result)

Search Status: POTENTIAL_MATCH

Potential Candidates:

Rank 1
Name: Test Person
Best Distance: 0.2426
Best Photo ID: 3
Reference Photos: 2
----------------------------------------
Rank 2
Name: Test Person 2
Best Distance: 0.9581
Best Photo ID: 2
Reference Photos: 1
----------------------------------------


In [26]:
def calculate_person_statistics(query_embedding):

    face_database = load_face_database()

    person_data = {}

    for record in face_database:

        distance = cosine_distance(
            query_embedding,
            record["embedding"]
        )

        person_id = record["person_id"]

        if person_id not in person_data:

            person_data[person_id] = {
                "person_id": person_id,
                "name": record["name"],
                "age": record["age"],
                "gender": record["gender"],
                "last_seen_location": record["last_seen_location"],
                "contact_information": record["contact_information"],
                "report_date": record["report_date"],
                "status": record["status"],
                "distances": [],
                "best_photo_id": record["photo_id"]
            }

        person_data[person_id]["distances"].append(distance)

        if distance < min(
            person_data[person_id]["distances"]
        ):
            person_data[person_id]["best_photo_id"] = record["photo_id"]

    results = []

    for person in person_data.values():

        distances = person["distances"]

        results.append({
            "person_id": person["person_id"],
            "name": person["name"],
            "age": person["age"],
            "gender": person["gender"],
            "last_seen_location": person["last_seen_location"],
            "contact_information": person["contact_information"],
            "report_date": person["report_date"],
            "status": person["status"],
            "best_distance": min(distances),
            "average_distance": np.mean(distances),
            "best_photo_id": person["best_photo_id"],
            "reference_photos": len(distances)
        })

    results.sort(
        key=lambda x: x["best_distance"]
    )

    return results


print("Updated person statistics function ready!")

Updated person statistics function ready!


In [27]:
query_embeddings_4 = generate_face_embeddings(
    "deepface_repo/tests/unit/dataset/img4.jpg"
)

query_embedding_4 = query_embeddings_4[0]

statistics = calculate_person_statistics(
    query_embedding_4
)

print("Person-Level Statistics:\n")

for rank, person in enumerate(statistics, start=1):

    print(f"Rank {rank}")
    print(f"Name: {person['name']}")
    print(f"Age: {person['age']}")
    print(f"Gender: {person['gender']}")
    print(f"Last Seen Location: {person['last_seen_location']}")
    print(f"Contact: {person['contact_information']}")
    print(f"Report Date: {person['report_date']}")
    print(f"Status: {person['status']}")
    print(f"Best Distance: {person['best_distance']:.4f}")
    print(f"Average Distance: {person['average_distance']:.4f}")
    print(f"Best Photo ID: {person['best_photo_id']}")
    print(f"Reference Photos: {person['reference_photos']}")
    print("-" * 50)

Person-Level Statistics:

Rank 1
Name: Test Person
Age: 25
Gender: Male
Last Seen Location: Delhi
Contact: test@example.com
Report Date: 2026-09-10
Status: MISSING
Best Distance: 0.2426
Average Distance: 0.3671
Best Photo ID: 1
Reference Photos: 2
--------------------------------------------------
Rank 2
Name: Test Person 2
Age: 30
Gender: Male
Last Seen Location: Mumbai
Contact: test2@example.com
Report Date: 2026-09-10
Status: MISSING
Best Distance: 0.9581
Average Distance: 0.9581
Best Photo ID: 2
Reference Photos: 1
--------------------------------------------------


In [20]:
# Calculate person-level statistics for the unseen query

query_embeddings_4 = generate_face_embeddings(
    "deepface_repo/tests/unit/dataset/img4.jpg"
)

query_embedding_4 = query_embeddings_4[0]

statistics = calculate_person_statistics(
    query_embedding_4
)

print("Person-Level Statistics:\n")

for rank, person in enumerate(statistics, start=1):

    print(
        f"{rank}. {person['name']} "
        f"| Best: {person['best_distance']:.4f} "
        f"| Average: {person['average_distance']:.4f} "
        f"| Photos: {person['reference_photos']}"
    )

Person-Level Statistics:

1. Test Person | Best: 0.2426 | Average: 0.3671 | Photos: 2
2. Test Person 2 | Best: 0.9581 | Average: 0.9581 | Photos: 1


In [28]:
def search_missing_person(image_path, top_k=5, threshold=0.68):

    # Generate query embedding
    query_embeddings = generate_face_embeddings(image_path)

    # No face detected
    if len(query_embeddings) == 0:
        return {
            "status": "NO_FACE",
            "results": []
        }

    # Multiple faces detected
    if len(query_embeddings) > 1:
        return {
            "status": "MULTIPLE_FACES",
            "results": []
        }

    # Single face
    query_embedding = query_embeddings[0]

    # Calculate person-level statistics
    results = calculate_person_statistics(
        query_embedding
    )

    # No registered people
    if len(results) == 0:
        return {
            "status": "NO_DATABASE_RECORDS",
            "results": []
        }

    # Keep top candidates
    results = results[:top_k]

    # Apply threshold
    if results[0]["best_distance"] < threshold:
        status = "POTENTIAL_MATCH"
    else:
        status = "NO_RELIABLE_MATCH"

    return {
        "status": status,
        "results": results
    }


print("Final production search function ready!")

Final production search function ready!


In [30]:
result = search_missing_person(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

print("Search Status:", result["status"])
print()

for rank, person in enumerate(result["results"], start=1):

    print(f"Rank {rank}")
    print(f"Name: {person['name']}")
    print(f"Age: {person['age']}")
    print(f"Gender: {person['gender']}")
    print(f"Last Seen Location: {person['last_seen_location']}")
    print(f"Contact: {person['contact_information']}")
    print(f"Report Date: {person['report_date']}")
    print(f"Status: {person['status']}")
    print(f"Best Distance: {person['best_distance']:.4f}")
    print(f"Average Distance: {person['average_distance']:.4f}")
    print(f"Best Photo ID: {person['best_photo_id']}")
    print(f"Reference Photos: {person['reference_photos']}")
    print("-" * 50)

Search Status: POTENTIAL_MATCH

Rank 1
Name: Test Person
Age: 25
Gender: Male
Last Seen Location: Delhi
Contact: test@example.com
Report Date: 2026-09-10
Status: MISSING
Best Distance: 0.2426
Average Distance: 0.3671
Best Photo ID: 1
Reference Photos: 2
--------------------------------------------------
Rank 2
Name: Test Person 2
Age: 30
Gender: Male
Last Seen Location: Mumbai
Contact: test2@example.com
Report Date: 2026-09-10
Status: MISSING
Best Distance: 0.9581
Average Distance: 0.9581
Best Photo ID: 2
Reference Photos: 1
--------------------------------------------------


In [22]:
result = search_missing_person(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

print("Search Status:", result["status"])
print()

for rank, person in enumerate(result["results"], start=1):

    print(
        f"{rank}. {person['name']} "
        f"| Best: {person['best_distance']:.4f} "
        f"| Average: {person['average_distance']:.4f} "
        f"| References: {person['reference_photos']}"
    )

Search Status: POTENTIAL_MATCH

1. Test Person | Best: 0.2426 | Average: 0.3671 | References: 2
2. Test Person 2 | Best: 0.9581 | Average: 0.9581 | References: 1


## Person Metadata in Search Results

A face match alone is not sufficient for a missing-person application.

The search result should also provide relevant information about the
candidate person, such as:

- Name
- Age
- Gender
- Last seen location
- Contact information
- Report date
- Missing-person status

This information will later be displayed to authorized users for
human verification.

The face-recognition system therefore acts as a candidate-ranking
component, while the stored personal information provides the
context required for verification.

In [23]:
cursor.execute("""
SELECT
    person_id,
    name,
    age,
    gender,
    last_seen_location,
    contact_information,
    report_date,
    status
FROM persons
""")

persons = cursor.fetchall()

print("Registered Persons:\n")

for person in persons:
    print(person)

Registered Persons:

(1, 'Test Person', 25, 'Male', 'Delhi', 'test@example.com', '2026-09-10', 'MISSING')
(2, 'Test Person 2', 30, 'Male', 'Mumbai', 'test2@example.com', '2026-09-10', 'MISSING')


In [31]:
def load_face_database():

    cursor.execute("""
    SELECT
        persons.person_id,
        persons.name,
        persons.age,
        persons.gender,
        persons.last_seen_location,
        persons.contact_information,
        persons.report_date,
        persons.status,
        photos.photo_id,
        photos.file_path,
        embeddings.embedding,
        embeddings.model_name
    FROM embeddings
    JOIN photos
        ON embeddings.photo_id = photos.photo_id
    JOIN persons
        ON photos.person_id = persons.person_id
    WHERE persons.status = 'MISSING'
    """)

    records = cursor.fetchall()

    database = []

    for record in records:

        embedding = np.frombuffer(
            record[10],
            dtype=np.float64
        )

        database.append({
            "person_id": record[0],
            "name": record[1],
            "age": record[2],
            "gender": record[3],
            "last_seen_location": record[4],
            "contact_information": record[5],
            "report_date": record[6],
            "status": record[7],
            "photo_id": record[8],
            "file_path": record[9],
            "embedding": embedding,
            "model_name": record[11]
        })

    return database


print("Missing-person database loader ready!")

Missing-person database loader ready!


In [32]:
face_database = load_face_database()

print("Active missing-person embeddings:", len(face_database))
print()

for record in face_database:
    print(
        f"Person ID: {record['person_id']} | "
        f"Name: {record['name']} | "
        f"Status: {record['status']} | "
        f"Photo ID: {record['photo_id']}"
    )

Active missing-person embeddings: 3

Person ID: 1 | Name: Test Person | Status: MISSING | Photo ID: 1
Person ID: 2 | Name: Test Person 2 | Status: MISSING | Photo ID: 2
Person ID: 1 | Name: Test Person | Status: MISSING | Photo ID: 3


In [33]:
result = search_missing_person(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

print("Search Status:", result["status"])
print()

for rank, person in enumerate(result["results"], start=1):

    print(f"Rank {rank}")
    print(f"Name: {person['name']}")
    print(f"Age: {person['age']}")
    print(f"Last Seen: {person['last_seen_location']}")
    print(f"Status: {person['status']}")
    print(f"Best Distance: {person['best_distance']:.4f}")
    print(f"Average Distance: {person['average_distance']:.4f}")
    print(f"Reference Photos: {person['reference_photos']}")
    print("-" * 40)

Search Status: POTENTIAL_MATCH

Rank 1
Name: Test Person
Age: 25
Last Seen: Delhi
Status: MISSING
Best Distance: 0.2426
Average Distance: 0.3671
Reference Photos: 2
----------------------------------------
Rank 2
Name: Test Person 2
Age: 30
Last Seen: Mumbai
Status: MISSING
Best Distance: 0.9581
Average Distance: 0.9581
Reference Photos: 1
----------------------------------------


In [36]:
cursor.execute("""
UPDATE persons
SET status = 'MISSING'
WHERE person_id = 2
""")

connection.commit()

print("Test Person 2 status updated to Missing.")

Test Person 2 status updated to Missing.


In [35]:
result = search_missing_person(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

print("Search Status:", result["status"])
print()

for rank, person in enumerate(result["results"], start=1):

    print(
        f"{rank}. {person['name']} "
        f"| Status: {person['status']} "
        f"| Best: {person['best_distance']:.4f}"
    )

Search Status: POTENTIAL_MATCH

1. Test Person | Status: MISSING | Best: 0.2426


In [25]:
face_database = load_face_database()

print("Total embeddings:", len(face_database))
print()

for record in face_database:

    print(
        f"Person: {record['name']} | "
        f"Age: {record['age']} | "
        f"Location: {record['last_seen_location']} | "
        f"Status: {record['status']} | "
        f"Photo ID: {record['photo_id']}"
    )

Total embeddings: 3

Person: Test Person | Age: 25 | Location: Delhi | Status: MISSING | Photo ID: 1
Person: Test Person 2 | Age: 30 | Location: Mumbai | Status: MISSING | Photo ID: 2
Person: Test Person | Age: 25 | Location: Delhi | Status: MISSING | Photo ID: 3


In [41]:
def format_search_results(result):

    formatted = {
        "status": result["status"],
        "candidates": []
    }

    for rank, person in enumerate(result["results"], start=1):

        candidate = {
            "rank": rank,

            "person": {
                "person_id": int(person["person_id"]),
                "name": person["name"],
                "age": int(person["age"]) if person["age"] is not None else None,
                "gender": person["gender"],
                "last_seen_location": person["last_seen_location"],
                "contact_information": person["contact_information"],
                "report_date": person["report_date"],
                "status": person["status"]
            },

            "match": {
                "best_distance": float(person["best_distance"]),
                "average_distance": float(person["average_distance"]),
                "best_photo_id": int(person["best_photo_id"]),
                "reference_photos": int(person["reference_photos"])
            }
        }

        formatted["candidates"].append(candidate)

    return formatted


print("JSON-safe result formatter ready!")

JSON-safe result formatter ready!


In [38]:
result = search_missing_person(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

formatted_result = format_search_results(result)

formatted_result

{'status': 'POTENTIAL_MATCH',
 'candidates': [{'rank': 1,
   'person': {'person_id': 1,
    'name': 'Test Person',
    'age': 25,
    'gender': 'Male',
    'last_seen_location': 'Delhi',
    'contact_information': 'test@example.com',
    'report_date': '2026-09-10',
    'status': 'MISSING'},
   'match': {'best_distance': np.float64(0.24257912362699152),
    'average_distance': np.float64(0.3670646166563755),
    'best_photo_id': 1,
    'reference_photos': 2}},
  {'rank': 2,
   'person': {'person_id': 2,
    'name': 'Test Person 2',
    'age': 30,
    'gender': 'Male',
    'last_seen_location': 'Mumbai',
    'contact_information': 'test2@example.com',
    'report_date': '2026-09-10',
    'status': 'MISSING'},
   'match': {'best_distance': np.float64(0.9580921306799807),
    'average_distance': np.float64(0.9580921306799807),
    'best_photo_id': 2,
    'reference_photos': 1}}]}

In [39]:
def search_missing_person_api(image_path, top_k=5, threshold=0.68):

    result = search_missing_person(
        image_path,
        top_k=top_k,
        threshold=threshold
    )

    formatted_result = format_search_results(result)

    return formatted_result


print("API-ready search function ready!")

API-ready search function ready!


In [40]:
api_result = search_missing_person_api(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

api_result

{'status': 'POTENTIAL_MATCH',
 'candidates': [{'rank': 1,
   'person': {'person_id': 1,
    'name': 'Test Person',
    'age': 25,
    'gender': 'Male',
    'last_seen_location': 'Delhi',
    'contact_information': 'test@example.com',
    'report_date': '2026-09-10',
    'status': 'MISSING'},
   'match': {'best_distance': np.float64(0.24257912362699152),
    'average_distance': np.float64(0.3670646166563755),
    'best_photo_id': 1,
    'reference_photos': 2}},
  {'rank': 2,
   'person': {'person_id': 2,
    'name': 'Test Person 2',
    'age': 30,
    'gender': 'Male',
    'last_seen_location': 'Mumbai',
    'contact_information': 'test2@example.com',
    'report_date': '2026-09-10',
    'status': 'MISSING'},
   'match': {'best_distance': np.float64(0.9580921306799807),
    'average_distance': np.float64(0.9580921306799807),
    'best_photo_id': 2,
    'reference_photos': 1}}]}

In [42]:
api_result = search_missing_person_api(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

print(api_result)

{'status': 'POTENTIAL_MATCH', 'candidates': [{'rank': 1, 'person': {'person_id': 1, 'name': 'Test Person', 'age': 25, 'gender': 'Male', 'last_seen_location': 'Delhi', 'contact_information': 'test@example.com', 'report_date': '2026-09-10', 'status': 'MISSING'}, 'match': {'best_distance': 0.24257912362699152, 'average_distance': 0.3670646166563755, 'best_photo_id': 1, 'reference_photos': 2}}, {'rank': 2, 'person': {'person_id': 2, 'name': 'Test Person 2', 'age': 30, 'gender': 'Male', 'last_seen_location': 'Mumbai', 'contact_information': 'test2@example.com', 'report_date': '2026-09-10', 'status': 'MISSING'}, 'match': {'best_distance': 0.9580921306799807, 'average_distance': 0.9580921306799807, 'best_photo_id': 2, 'reference_photos': 1}}]}


In [43]:
print("=== Production Search Tests ===")
print()

# Test 1: Normal matching image
result_single = search_missing_person_api(
    "deepface_repo/tests/unit/dataset/img4.jpg"
)

print("Test 1 - Single Face")
print("Status:", result_single["status"])
print("Top Candidate:", result_single["candidates"][0]["person"]["name"])
print()

# Test 2: No face
result_no_face = search_missing_person_api(
    "test_no_face.jpg"
)

print("Test 2 - No Face")
print("Status:", result_no_face["status"])
print()

# Test 3: Multiple faces
result_multiple = search_missing_person_api(
    "test_multiple_faces.jpg"
)

print("Test 3 - Multiple Faces")
print("Status:", result_multiple["status"])
print()

# Test 4: Unknown person
result_unknown = search_missing_person_api(
    "deepface_repo/tests/unit/dataset/img13.jpg"
)

print("Test 4 - Unknown / Unregistered Person")
print("Status:", result_unknown["status"])
print()

print("=== Tests Completed ===")

=== Production Search Tests ===

Test 1 - Single Face
Status: POTENTIAL_MATCH
Top Candidate: Test Person

Test 2 - No Face
Status: NO_FACE

Test 3 - Multiple Faces
Status: MULTIPLE_FACES

Test 4 - Unknown / Unregistered Person
Status: NO_RELIABLE_MATCH

=== Tests Completed ===


## Phase 7 Results and Conclusion

### Production Search Engine

The production face-search engine was successfully implemented by integrating:

- MTCNN face detection
- ArcFace 512-dimensional face embeddings
- SQLite-based face embedding storage
- Cosine-distance comparison
- Person-level candidate ranking
- Best and average distance calculation
- Missing-person status filtering
- Threshold-based potential-match detection
- Structured JSON-safe search results

### Search Pipeline

```text
Query Photograph
       |
       v
MTCNN Face Detection
       |
       v
ArcFace Embedding
       |
       v
SQLite Face Database
       |
       v
Only MISSING Persons
       |
       v
Cosine Distance
       |
       v
Person-Level Ranking
       |
       v
Best + Average Distance
       |
       v
Threshold Decision
       |
       v
Potential Candidates
       |
       v
Human Verification